In [ ]:
#https://data.mendeley.com/datasets/jnxh2zzynv/2
#Jiang, jiaming (2024), “Japanese corporates GX patents”, Mendeley Data, V2, doi: 10.17632/jnxh2zzynv.2

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)

df = pd.read_pickle("patents_clean.pkl")
df["pub_year"] = df["pub_date"].dt.year
df["gap"] = (df["pub_date"] - df["filing"]).dt.days / 365.25
df["doc_type"] = df["Document number"].str.extract(r"^(特許|特開|再表|特表)")[0]

# 1. 공개일이 어느 문헌의 날짜인지: 문헌 유형별 출원~공개 간격
print(df.groupby("doc_type")["gap"].describe().round(2).to_string())

# 2. 공개연도별 건수, 기업 수, 상위 4개사 비중, WO 비중
df["is_wo"] = df["Publication number"].fillna("").str.startswith("WO")
top4 = ["トヨタ自動車", "三菱電機", "日産自動車", "ホンダ"]
g = df.groupby("pub_year").agg(
    n=("app_key", "size"),
    firms=("corporate", "nunique"),
    top4_share=("corporate", lambda s: s.isin(top4).mean()),
    wo_share=("is_wo", "mean"),
    filing_min=("filing", lambda s: s.dt.year.min()),
    filing_med=("filing", lambda s: s.dt.year.median()),
).round(3)
print(g.to_string())

# 3. 분석 창(공개 2006~2015)의 기업 규모
w = df[(df["pub_date"] >= "2006-01-01") & (df["pub_date"] <= "2015-12-31")]
print(f"\n창 안: {len(w):,}건 ({len(w)/len(df):.1%}), 기업 {w['corporate'].nunique()}곳")
c = w["corporate"].value_counts()
for m in (5, 10, 20, 30, 50):
    k = c[c >= m]
    print(f"창 내 {m}건 이상: 기업 {len(k)}곳, 특허 커버 {k.sum()/len(w):.1%}")
print("창 안에서 WO 비중:", round(w["is_wo"].mean(), 3))

In [ ]:
import unicodedata
import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

df = pd.read_pickle("patents_clean.pkl")
df["pub_year"] = df["pub_date"].dt.year
df["corp_norm"] = df["corporate"].map(lambda s: unicodedata.normalize("NFKC", str(s)).strip())
df["is_wo"] = df["Publication number"].fillna("").str.startswith("WO")
print("전각/반각 통일로 합쳐진 기업 수:", df["corporate"].nunique() - df["corp_norm"].nunique())

# ---------- 1. 창 표본 ----------
w = df[(df["pub_date"] >= "2006-01-01") & (df["pub_date"] <= "2015-12-31")].copy()
print(f"창: {len(w):,}건, 기업 {w['corp_norm'].nunique()}곳")

# ---------- 2. 기업 마스터 ----------
firm = (w.groupby("corp_norm").agg(
            corp_ja=("corporate", "first"),
            industry=("industry", lambda s: s.mode().iat[0]),
            n_win=("app_key", "size"),
            n_2010_15=("pub_year", lambda s: (s >= 2010).sum()),
            n_wo=("is_wo", "sum"),
            first_pub=("pub_year", "min"),
            n_pubyears=("pub_year", "nunique"))
          .reset_index().sort_values("n_win", ascending=False))
firm["early_share"] = (1 - firm["n_2010_15"] / firm["n_win"]).round(3)
firm["tier"] = pd.cut(firm["n_win"], [0, 9, 29, np.inf], labels=["<10", "10-29", "30+"])

print("\n[기업 등급별]")
print(firm.groupby("tier", observed=True).agg(firms=("corp_norm", "size"),
                                              patents=("n_win", "sum")).to_string())
print("\n10건 이상 기업 중 2010년 이후에야 처음 등장한 기업:",
      int(((firm["n_win"] >= 10) & (firm["first_pub"] >= 2010)).sum()), "/",
      int((firm["n_win"] >= 10).sum()))
print("\n[상위 20개 기업]")
print(firm.head(20)[["corp_ja", "industry", "n_win", "n_wo", "first_pub", "early_share"]].to_string(index=False))

# ---------- 3. FI 분류 점검 (기술분야 정의용) ----------
fi = w["FI"].fillna("")
print("\n[FI 컬럼 예시]")
print(fi.sample(10, random_state=0).to_string())

w["fi_sub"] = fi.str.findall(r"[A-H]\d{2}[A-Z]")     # 서브클래스(예: B60K, H01M)
ex = w[["app_key", "corp_norm", "pub_year", "fi_sub"]].explode("fi_sub").dropna(subset=["fi_sub"])
print("\nFI가 비어 있거나 서브클래스 추출 실패한 비율:", round((w["fi_sub"].str.len() == 0).mean(), 3))
print("\n[서브클래스 상위 30 (특허 수 기준, 중복 제거)]")
print(ex.drop_duplicates(["app_key", "fi_sub"])["fi_sub"].value_counts().head(30).to_string())

# ---------- 4. 저장 ----------
w.drop(columns=["keys"], errors="ignore").to_pickle("patents_window.pkl")
firm.to_csv("firm_master_window.csv", index=False, encoding="utf-8-sig")
print("\nsaved patents_window.pkl:", len(w), "| firm_master_window.csv:", len(firm))

In [ ]:
import re
import numpy as np
import pandas as pd
pd.set_option("display.width", 200)

w = pd.read_pickle("patents_window.pkl")

# ---------- 1. FI -> (서브클래스+메인그룹) 코드 목록, 예: H01M8, B60K6 ----------
def fi_groups(s):
    return re.findall(r"([A-H]\d{2}[A-Z])(\d+)/", s)          # [('H01M','8'), ...]

LIGHT_GROUPS = {"33", "37", "44", "45", "47"}                 # H05B 중 조명·OLED
def domain_of(sub, grp):
    if sub == "H01M":
        return "FUELCELL" if grp == "8" else "BATTERY"
    if sub in {"B60L", "B60K", "B60W", "F02D", "F16H", "B62D", "B60R"}:
        return "VEHICLE"
    if sub == "H02S" or (sub == "H01L" and grp == "31"):
        return "SOLAR"
    if sub in {"H02J", "H02M", "H02K", "H01G", "H01L"}:
        return "POWER"
    if sub == "H05B":
        return "LIGHT" if grp in LIGHT_GROUPS else "THERMAL"
    if sub in {"H10K", "F21Y", "F21S", "F21V"} or (sub == "H01L" and grp == "33"):
        return "LIGHT"
    if sub in {"F24H", "F25B", "F24F", "F01K", "F28D", "F28F"}:
        return "THERMAL"
    if sub.startswith("C08") or sub in {"C01B", "C09K", "B32B", "B01J", "H01B"}:
        return "MATERIAL"
    return "OTHER"

def doms(s):
    seen = []
    for sub, grp in fi_groups(s):
        d = domain_of(sub, grp)
        if d not in seen:
            seen.append(d)
    return seen                                               # FI 기재 순서 유지

w["domains"] = w["FI"].fillna("").map(doms)
w["primary"] = w["domains"].map(lambda l: next((d for d in l if d != "OTHER"), "OTHER"))
w["n_dom"] = w["domains"].map(lambda l: len([d for d in l if d != "OTHER"]))

# ---------- 2. 분야 분포 점검 ----------
ALL = ["FUELCELL", "BATTERY", "VEHICLE", "POWER", "SOLAR", "LIGHT", "THERMAL", "MATERIAL"]
print("[분야별 특허 수] 복수 소속(multi) vs 대표 분야(primary)")
multi = {d: int(w["domains"].map(lambda l: d in l).sum()) for d in ALL}
chk = pd.DataFrame({"multi": multi, "primary": w["primary"].value_counts()}).fillna(0).astype(int)
print(chk.reindex(ALL + ["OTHER"]).to_string())
print("분야 미배정(OTHER만):", int((w["n_dom"] == 0).sum()), f"({(w['n_dom'] == 0).mean():.1%})")
print("2개 이상 분야에 걸친 특허 비중:", round((w["n_dom"] >= 2).mean(), 3))

# ---------- 3. 기업 x 분야 표 (창 누적) ----------
firms10 = w.groupby("corp_norm").size()
firms10 = firms10[firms10 >= 10].index
ex = w[w["corp_norm"].isin(firms10)][["app_key", "corp_norm", "pub_year", "is_wo", "domains"]].explode("domains")
ex = ex[ex["domains"].isin(ALL)]
fd = (ex.groupby(["corp_norm", "domains"]).agg(n_pat=("app_key", "nunique"), n_wo=("is_wo", "sum"))
        .reset_index().rename(columns={"domains": "domain"}))

# 빠진 칸은 0으로 채운 완전한 격자
grid = pd.MultiIndex.from_product([firms10, ALL], names=["corp_norm", "domain"])
fd = fd.set_index(["corp_norm", "domain"]).reindex(grid, fill_value=0).reset_index()

print(f"\n기업 {len(firms10)}곳 x 분야 {len(ALL)} = {len(fd)}칸")
print("특허 0건인 칸 비율:", round((fd["n_pat"] == 0).mean(), 3))
for m in (1, 5, 10):
    print(f"특허 {m}건 이상인 칸: {(fd['n_pat'] >= m).sum()}")
print("\n[분야별로 특허 5건 이상인 기업 수]")
print(fd[fd["n_pat"] >= 5].groupby("domain")["corp_norm"].nunique().reindex(ALL).to_string())
print("\n[기업별 분야 수(5건 이상) 분포]")
print(fd[fd["n_pat"] >= 5].groupby("corp_norm")["domain"].nunique().value_counts().sort_index().to_string())

# ---------- 4. 저장 ----------
w.to_pickle("patents_window_domain.pkl")
fd.to_csv("firm_domain_patent.csv", index=False, encoding="utf-8-sig")
print("\nsaved patents_window_domain.pkl, firm_domain_patent.csv")

In [ ]:
import os, json, time, hashlib, unicodedata
import requests
import pandas as pd

# ============================================================ 설정
STAGE = "collect"                                # "match" -> 수작업 확인 -> "collect"
MAILTO = "<REDACTED_EMAIL>"
OA_KEY = os.environ.get("OPENALEX_API_KEY", "")  # 환경변수 사용 (코드에 직접 쓰지 마세요)
OA_KEY = "<REDACTED_API_KEY>"
FIRM_MASTER = "firm_master_window.csv"           # 이전 단계 산출물
MIN_PATENTS = 10                                 # 창 내 특허 10건 이상 기업(73곳)
Y0, Y1 = 2006, 2025
PERIODS = {"base_2006_2015": (2006, 2015), "out_2016_2025": (2016, 2025)}
TYPES = "type:article|review,is_paratext:false"  # 논문(연구논문+리뷰)만
OUT, CACHE = "oa_out", os.path.join("oa_out", "cache")
os.makedirs(CACHE, exist_ok=True)
BASE = "https://api.openalex.org"
NCALL = 0

# 영문 힌트 (매칭 정확도용). 없는 기업은 일본어 이름으로만 검색
HINT = {"トヨタ自動車": "toyota", "三菱電機": "mitsubishi electric", "日産自動車": "nissan",
        "ホンダ": "honda", "コニカミノルタ": "konica minolta", "凸版印刷": "toppan",
        "三菱自動車": "mitsubishi motors", "ノーリツ": "noritz", "日本特殊陶業": "ngk spark plug",
        "富士電機": "fuji electric", "住友電気工業": "sumitomo electric", "東レ": "toray",
        "TOTO": "toto", "リンナイ": "rinnai", "出光興産": "idemitsu", "スズキ": "suzuki",
        "信越化学工業": "shin-etsu", "三菱マテリアル": "mitsubishi materials",
        "中国電力": "chugoku electric", "日本電気硝子": "nippon electric glass",
        "三井化学": "mitsui chemicals", "川崎重工業": "kawasaki heavy", "フジクラ": "fujikura"}


# ============================================================ API 공통
def oa_get(path, params, tries=6):
    global NCALL
    assert OA_KEY, "OA_KEY가 비어 있습니다 (환경변수 OPENALEX_API_KEY 설정)"
    ck = hashlib.md5((path + json.dumps(params, sort_keys=True)).encode()).hexdigest()
    fn = os.path.join(CACHE, ck + ".json")
    if os.path.exists(fn):                                   # 캐시: 재실행해도 호출 안 함
        return json.load(open(fn, encoding="utf-8"))
    p = dict(params); p["mailto"] = MAILTO; p["api_key"] = OA_KEY
    for t in range(tries):
        try:
            r = requests.get(BASE + path, params=p, timeout=90)
        except requests.RequestException:
            time.sleep(2 ** t); continue
        if r.status_code == 200:
            NCALL += 1
            j = r.json(); json.dump(j, open(fn, "w", encoding="utf-8"), ensure_ascii=False)
            time.sleep(0.12); return j
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(min(60, 2 * 2 ** t)); continue
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:300]}")
    raise RuntimeError("재시도 초과(429면 일일 한도 소진 가능): " + path)


def group(filt, key, path="/works"):
    """group_by 집계 -> [(키, 이름, 건수)]. API는 1페이지(최대 200개 버킷)만 허용."""
    # "page": 1 은 기존 캐시 키와 맞추기 위해 유지 (이미 받은 응답 재사용)
    j = oa_get(path, {"filter": filt, "group_by": key, "per_page": 200, "page": 1})
    g = j.get("group_by", [])
    if len(g) >= 200:
        print(f"  [주의] 버킷 200개 도달(잘렸을 수 있음): {key} | {filt[:80]}")
    return [(x["key"].split("/")[-1], x.get("key_display_name", ""), x["count"]) for x in g]


def topics_full(base, a, b):
    """주제별 논문 수. 200개를 넘으면 분야(field)별로 나눠서 합친다."""
    flt = f"{base},publication_year:{a}-{b}"
    rows = group(flt, "primary_topic.id")
    if len(rows) < 200:
        return rows
    out = []
    for fid, _, _ in group(flt, "primary_topic.field.id"):
        out += group(f"{flt},primary_topic.field.id:{fid}", "primary_topic.id")
    return out


def firms():
    f = pd.read_csv(FIRM_MASTER)
    return f[f["n_win"] >= MIN_PATENTS][["corp_norm", "corp_ja", "n_win"]].reset_index(drop=True)


# ============================================================ 1단계: 기업 -> 기관 매칭
def search_inst(q):
    j = oa_get("/institutions", {"search": q, "filter": "country_code:JP", "per_page": 8,
                                  "select": "id,display_name,type,works_count,ror"})
    return [dict(query=q, id=r["id"].split("/")[-1], display_name=r["display_name"], type=r.get("type"),
                 works_count=r.get("works_count", 0), ror=r.get("ror")) for r in j.get("results", [])]


def do_match():
    F = firms(); cand_rows, map_rows = [], []
    for _, r in F.iterrows():
        hint = HINT.get(r["corp_norm"]) or HINT.get(r["corp_ja"])
        rows = search_inst(r["corp_ja"]) + (search_inst(hint) if hint else [])
        c = pd.DataFrame(rows).drop_duplicates("id") if rows else pd.DataFrame(
            columns=["query", "id", "display_name", "type", "works_count", "ror"])
        c.insert(0, "corp_norm", r["corp_norm"]); cand_rows.append(c)
        comp = c[c["type"] == "company"]; pick, status = None, "NONE(수동 필요)"
        if hint and len(comp):
            h = comp[comp["display_name"].str.lower().str.contains(hint, regex=False)]
            if len(h): pick, status = h.sort_values("works_count", ascending=False).iloc[0], "hint일치(검토)"
        if pick is None and len(comp):
            pick, status = comp.iloc[0], "약함(반드시 검토)"
        map_rows.append(dict(corp_norm=r["corp_norm"], corp_ja=r["corp_ja"], n_win=r["n_win"],
                             openalex_ids=pick["id"] if pick is not None else "",
                             picked_name=pick["display_name"] if pick is not None else "", status=status))
    C = pd.concat(cand_rows, ignore_index=True); M = pd.DataFrame(map_rows)
    C.to_csv("oa_inst_candidates.csv", index=False, encoding="utf-8-sig")
    M.to_csv("oa_inst_map_draft.csv", index=False, encoding="utf-8-sig")
    print(f"API 호출 {NCALL}회 (캐시 제외)")
    print(M["status"].value_counts().to_string())
    print("\n[초안 지도]"); print(M[["corp_ja", "n_win", "picked_name", "openalex_ids", "status"]].to_string(index=False))
    print("\n다음: oa_inst_candidates.csv를 보며 oa_inst_map_draft.csv의 openalex_ids를 수정하고 "
          "oa_inst_map.csv로 저장하세요. 계열 연구소는 ID를 | 로 이어 쓰세요 (예: I111|I222). "
          "처리하지 않을 기업은 openalex_ids를 비워 두세요.")


# ============================================================ 2단계: 논문 집계
def collect_firm(ids, univ_flt):
    base = f"authorships.institutions.lineage:{ids},{TYPES}"
    yr = f"publication_year:{Y0}-{Y1}"
    tot = {int(k): c for k, _, c in group(f"{base},{yr}", "publication_year")}
    uni = {int(k): c for k, _, c in group(f"{base},{univ_flt},{yr}", "publication_year")}
    sub = []
    for y in range(Y0, Y1 + 1):
        if tot.get(y, 0) == 0: continue
        sub += [(y, k, n, c) for k, n, c in group(f"{base},publication_year:{y}", "primary_topic.subfield.id")]
    top = []
    for per, (a, b) in PERIODS.items():
        top += [(per, k, n, c) for k, n, c in topics_full(base, a, b)]
    return tot, uni, sub, top


def pick_univ_filter(ids):
    """대학 공저 필터 문법 점검 (스키마가 다르면 대체 문법 시도)."""
    for flt in ("authorships.institutions.type:education", "institutions.type:education"):
        try:
            oa_get("/works", {"filter": f"authorships.institutions.lineage:{ids},{flt},{TYPES}", "per_page": 1})
            return flt
        except RuntimeError as e:
            print("  필터 실패:", flt, str(e)[:120])
    raise SystemExit("대학 공저 필터를 찾지 못했습니다. 위 오류 메시지를 알려 주세요.")


def do_collect():
    M = pd.read_csv("oa_inst_map.csv", dtype=str).fillna("")
    M = M[M["openalex_ids"] != ""].reset_index(drop=True)
    print("수집 대상 기업:", len(M))
    univ_flt = pick_univ_filter(M["openalex_ids"].iloc[0]); print("대학 공저 필터:", univ_flt)
    fy, fs, ft = [], [], []
    for i, r in M.iterrows():
        tot, uni, sub, top = collect_firm(r["openalex_ids"], univ_flt)
        for y in range(Y0, Y1 + 1):
            fy.append(dict(corp_norm=r["corp_norm"], year=y, n_works=tot.get(y, 0), n_univ_coauth=uni.get(y, 0)))
        fs += [dict(corp_norm=r["corp_norm"], year=y, subfield_id=k, subfield=n, n=c) for y, k, n, c in sub]
        ft += [dict(corp_norm=r["corp_norm"], period=p, topic_id=k, topic=n, n=c) for p, k, n, c in top]
        print(f"[{i + 1}/{len(M)}] {r['corp_ja']:<10} 총 {sum(tot.values()):>6,}편 | 호출 누적 {NCALL}")
    FY, FS, FT = pd.DataFrame(fy), pd.DataFrame(fs), pd.DataFrame(ft)
    FY.to_csv(os.path.join(OUT, "oa_firm_year.csv"), index=False, encoding="utf-8-sig")
    FS.to_csv(os.path.join(OUT, "oa_firm_year_subfield.csv"), index=False, encoding="utf-8-sig")
    FT.to_csv(os.path.join(OUT, "oa_firm_period_topic.csv"), index=False, encoding="utf-8-sig")

    # 주제(topic) -> 분야 계층 (8개 분야 대응용)
    tids, rows = sorted(FT["topic_id"].unique()), []
    for i in range(0, len(tids), 50):
        try:
            j = oa_get("/topics", {"filter": "openalex:" + "|".join(tids[i:i + 50]), "per_page": 50,
                                    "select": "id,display_name,subfield,field,domain,keywords"})
        except RuntimeError as e:
            print("topics 조회 실패:", str(e)[:150]); break
        for t in j.get("results", []):
            rows.append(dict(topic_id=t["id"].split("/")[-1], topic=t["display_name"],
                             subfield=(t.get("subfield") or {}).get("display_name"),
                             field=(t.get("field") or {}).get("display_name"),
                             domain=(t.get("domain") or {}).get("display_name"),
                             keywords="; ".join(t.get("keywords") or [])))
    pd.DataFrame(rows).to_csv(os.path.join(OUT, "oa_topic_map.csv"), index=False, encoding="utf-8-sig")

    # ---------- 진단 요약 ----------
    P = FY.assign(per=lambda d: pd.cut(d["year"], [2005, 2015, 2025], labels=["base", "out"])) \
          .groupby(["corp_norm", "per"], observed=True)[["n_works", "n_univ_coauth"]].sum().unstack(fill_value=0)
    P.columns = [f"{a}_{b}" for a, b in P.columns]
    for m in (1, 20, 100):
        print(f"기준선·결과 모두 {m}편 이상 기업: {int(((P['n_works_base'] >= m) & (P['n_works_out'] >= m)).sum())}곳")
    print("기준선 논문 0편 기업:", int((P["n_works_base"] == 0).sum()), "곳")
    print(f"대학 공저 비중(전체): {P['n_univ_coauth_out'].sum() / max(1, P['n_works_out'].sum()):.1%} (결과기간)")
    print("\n[논문 상위 15개 기업]")
    print(P.sort_values("n_works_out", ascending=False).head(15).to_string())
    print("\n[연도별 총 논문 수 (2025년은 색인 미완 가능)]")
    print(FY.groupby("year")["n_works"].sum().to_string())
    print(f"\n총 API 호출(캐시 제외): {NCALL}회 | 저장 폴더: {os.path.abspath(OUT)}")


if STAGE == "match":
    do_match()
else:
    do_collect()

In [ ]:
#

In [ ]:
import os, json, hashlib, re
import requests
import numpy as np
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 50)

OUT = "oa_out"; CACHE = os.path.join(OUT, "cache"); BASE = "https://api.openalex.org"
MAILTO = "<REDACTED_EMAIL>"
OA_KEY = os.environ.get("OPENALEX_API_KEY", "")      # 캐시만 쓰면 비어 있어도 됨

def oa_get(path, params):
    ck = hashlib.md5((path + json.dumps(params, sort_keys=True)).encode()).hexdigest()
    fn = os.path.join(CACHE, ck + ".json")
    if os.path.exists(fn):
        return json.load(open(fn, encoding="utf-8"))
    assert OA_KEY, "캐시에 없는 호출입니다. 환경변수 OPENALEX_API_KEY를 설정하세요"
    p = dict(params); p["mailto"] = MAILTO; p["api_key"] = OA_KEY
    r = requests.get(BASE + path, params=p, timeout=90); r.raise_for_status()
    j = r.json(); json.dump(j, open(fn, "w", encoding="utf-8"), ensure_ascii=False)
    return j

FY = pd.read_csv(os.path.join(OUT, "oa_firm_year.csv"))
FT = pd.read_csv(os.path.join(OUT, "oa_firm_period_topic.csv"), dtype={"topic_id": str})

# ---------- 1. 주제 -> 분야 계층 ----------
tids, rows = sorted(FT["topic_id"].unique()), []
for i in range(0, len(tids), 50):
    j = oa_get("/topics", {"filter": "openalex:" + "|".join(tids[i:i + 50]), "per_page": 50,
                            "select": "id,display_name,subfield,field,domain,keywords"})
    for t in j.get("results", []):
        kw = [(k.get("display_name") or k.get("keyword") or "") if isinstance(k, dict) else str(k)
              for k in (t.get("keywords") or [])]
        rows.append(dict(topic_id=t["id"].split("/")[-1], topic=t["display_name"],
                         subfield=(t.get("subfield") or {}).get("display_name"),
                         field=(t.get("field") or {}).get("display_name"),
                         domain=(t.get("domain") or {}).get("display_name"),
                         keywords="; ".join(kw)))
TM = pd.DataFrame(rows)
TM.to_csv(os.path.join(OUT, "oa_topic_map.csv"), index=False, encoding="utf-8-sig")
print("주제 수:", len(TM), "/ 집계에 나온 주제:", len(tids))

# ---------- 2. 진단 요약 ----------
P = FY.assign(per=lambda d: pd.cut(d["year"], [2005, 2015, 2025], labels=["base", "out"])) \
      .groupby(["corp_norm", "per"], observed=True)[["n_works", "n_univ_coauth"]].sum().unstack(fill_value=0)
P.columns = [f"{a}_{b}" for a, b in P.columns]
for m in (1, 20, 100):
    print(f"기준선·결과 모두 {m}편 이상 기업: {int(((P['n_works_base'] >= m) & (P['n_works_out'] >= m)).sum())}곳")
print("기준선 논문 0편 기업:", int((P["n_works_base"] == 0).sum()), "곳")
print(f"대학 공저 비중(결과기간): {P['n_univ_coauth_out'].sum() / max(1, P['n_works_out'].sum()):.1%}")

# ---------- 3. 주제 -> 8개 분야 (주제명만 사용, 앞에서 맞는 규칙 우선) ----------
EXCLUDE = r"embrittlement|corrosion"          # 키워드가 걸려도 GX 분야가 아닌 주제
RULES = [
 ("FUELCELL", r"fuel cell|solid oxide|proton exchange|oxygen reduction|electrolyz|water electrolysis|hydrogen storage|hydrogen production|electrocatalys"),
 ("SOLAR",    r"solar|photovoltaic|perovskite|dye-sensitized"),
 ("BATTERY",  r"batter|lithium|sodium-ion|supercapacitor|electrochemical energy|energy storage"),
 ("LIGHT",    r"light-emitting|\bleds?\b|\boleds?\b|organic electronics|luminescen|phosphor|lighting|quantum dot"),
 ("VEHICLE",  r"vehicle|automotive|hybrid electric|\bengines?\b|powertrain|traction|electric motor|motor drive|transmission"),
 ("THERMAL",  r"\bheat|thermal|thermoelectric|refrigerat|cooling|boiler|turbine|combustion|air condition"),
 ("POWER",    r"power system|power electronics|power converter|inverter|smart grid|power grid|microgrid|wide bandgap|silicon carbide|\bgan\b|gan-based|dc-dc|wireless power|electric machine"),
 ("MATERIAL", r"polymer|composite|ceramic|catalys|graphene|carbon nano|membrane|cellulose|zeolite|mesoporous|nanoparticle"),
]
def assign(topic):
    if re.search(EXCLUDE, topic or "", re.I):
        return "OTHER"
    for dom, pat in RULES:
        if re.search(pat, topic or "", re.I):
            return dom
    return "OTHER"

TM["gx_domain"] = [assign(t) for t in TM["topic"]]
tot = FT.groupby("topic_id")["n"].sum().rename("n_total")
TM = TM.merge(tot, left_on="topic_id", right_index=True, how="left")
TM.sort_values("n_total", ascending=False).to_csv(
    os.path.join(OUT, "oa_topic_domain.csv"), index=False, encoding="utf-8-sig")

print("\n[분야별 논문 수 합계 / 주제 수]")
print(TM.groupby("gx_domain").agg(papers=("n_total", "sum"), topics=("topic_id", "size")).to_string())
print("\n[OTHER 중 논문 많은 주제 40개: 빠진 분야가 없는지 확인]")
print(TM[TM["gx_domain"] == "OTHER"].sort_values("n_total", ascending=False)
        .head(40)[["topic", "subfield", "n_total"]].to_string(index=False))
print("\n[배정된 주제 중 논문 많은 60개: 오배정 확인]")
print(TM[TM["gx_domain"] != "OTHER"].sort_values("n_total", ascending=False)
        .head(60)[["topic", "gx_domain", "n_total"]].to_string(index=False))

In [ ]:
#

In [ ]:
import os
import pandas as pd
pd.set_option("display.width", 200)

OUT = "oa_out"
TM = pd.read_csv(os.path.join(OUT, "oa_topic_domain.csv"), dtype={"topic_id": str})

OVERRIDE = {                      # 주제명(정확히 일치) -> 분야
    "High-Voltage Power Transmission Systems": "POWER",
    "Magnetic and transport properties of perovskites and related materials": "OTHER",
    "Automotive and Human Injury Biomechanics": "OTHER",
    "Autonomous Vehicle Technology and Safety": "OTHER",
    "Building Energy and Comfort Optimization": "THERMAL",
    "Catalytic Processes in Materials Science": "MATERIAL",
    "High voltage insulation and dielectric phenomena": "POWER",
}
hit = TM["topic"].isin(OVERRIDE)
print("적용된 주제 수:", int(hit.sum()), "/", len(OVERRIDE))
print("못 찾은 주제:", [k for k in OVERRIDE if k not in set(TM["topic"])])
TM.loc[hit, "gx_domain"] = TM.loc[hit, "topic"].map(OVERRIDE)

print("\n[분야별 논문 수]")
print(TM.groupby("gx_domain").agg(papers=("n_total", "sum"), topics=("topic_id", "size")).to_string())
TM.to_csv(os.path.join(OUT, "oa_topic_domain_checked.csv"), index=False, encoding="utf-8-sig")
print("saved oa_topic_domain_checked.csv")

In [ ]:
#

In [ ]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
pd.set_option("display.width", 200)

OUT = "oa_out"
ALL = ["FUELCELL", "BATTERY", "VEHICLE", "POWER", "SOLAR", "LIGHT", "THERMAL", "MATERIAL"]

TMf = os.path.join(OUT, "oa_topic_domain_checked.csv")
TM = pd.read_csv(TMf if os.path.exists(TMf) else os.path.join(OUT, "oa_topic_domain.csv"), dtype={"topic_id": str})
print("분야 대응표:", os.path.basename(TMf if os.path.exists(TMf) else "oa_topic_domain.csv (미검토 초안)"))

FT = pd.read_csv(os.path.join(OUT, "oa_firm_period_topic.csv"), dtype={"topic_id": str})
M = pd.read_csv("oa_inst_map.csv", dtype=str).fillna("")
M = M[M["openalex_ids"] != ""][["corp_norm", "corp_ja", "n_win"]]
M["n_win"] = M["n_win"].astype(float)
fd = pd.read_csv("firm_domain_patent.csv")

# ---------- 논문: 기업 x 분야 x 기간 ----------
pp = FT.merge(TM[["topic_id", "gx_domain"]], on="topic_id", how="left")
pp = pp[pp["gx_domain"].isin(ALL)]
pv = (pp.pivot_table(index=["corp_norm", "gx_domain"], columns="period", values="n", aggfunc="sum", fill_value=0)
        .rename(columns={"base_2006_2015": "n_base", "out_2016_2025": "n_out"}).reset_index()
        .rename(columns={"gx_domain": "domain"}))

# ---------- 완전한 격자 + 특허 병합 ----------
grid = pd.MultiIndex.from_product([M["corp_norm"], ALL], names=["corp_norm", "domain"]).to_frame(index=False)
d = (grid.merge(pv, on=["corp_norm", "domain"], how="left")
         .merge(fd[["corp_norm", "domain", "n_pat", "n_wo"]], on=["corp_norm", "domain"], how="left")
         .merge(M, on="corp_norm", how="left")).fillna({"n_base": 0, "n_out": 0, "n_pat": 0, "n_wo": 0})
for c in ("n_base", "n_out", "n_pat", "n_wo"):
    d[c] = d[c].astype(float)
d["ln_pat"] = np.log1p(d["n_pat"]); d["ln_base"] = np.log1p(d["n_base"])
d.to_csv("panel_firm_domain.csv", index=False, encoding="utf-8-sig")

# ---------- 파일럿 판정 ----------
print(f"\n기업 {d['corp_norm'].nunique()}곳 x 분야 {len(ALL)} = {len(d)}칸")
print("특허 5건 이상 칸:", int((d["n_pat"] >= 5).sum()),
      "| 그중 결과 논문 1편 이상:", int(((d["n_pat"] >= 5) & (d["n_out"] >= 1)).sum()),
      "| 5편 이상:", int(((d["n_pat"] >= 5) & (d["n_out"] >= 5)).sum()))
print("결과 논문 합계가 0인 기업(모형에서 제외):", int((d.groupby("corp_norm")["n_out"].sum() == 0).sum()), "곳")
print("\n[분야별 합계]"); print(d.groupby("domain")[["n_pat", "n_base", "n_out"]].sum().reindex(ALL).to_string())
print("\n[분야별 결과 논문이 1편 이상인 기업 수]")
print(d[d["n_out"] > 0].groupby("domain")["corp_norm"].nunique().reindex(ALL).to_string())

# ---------- PPML (기업·분야 고정효과, 기업 단위 클러스터 표준오차) ----------
def ppml(data, formula, label):
    x = data[data.groupby("corp_norm")["n_out"].transform("sum") > 0]      # 결과가 전부 0인 기업은 고정효과가 식별 불가
    x = x[x.groupby("domain")["n_out"].transform("sum") > 0]
    g = pd.factorize(x["corp_norm"])[0]
    r = smf.glm(formula, data=x, family=sm.families.Poisson()).fit(
        cov_type="cluster", cov_kwds={"groups": g}, maxiter=200)
    out = {"model": label, "firms": x["corp_norm"].nunique(), "cells": len(x)}
    for v in ("ln_pat", "ln_base"):
        if v in r.params.index:
            out[v] = f"{r.params[v]:.3f} (se {r.bse[v]:.3f}, p={r.pvalues[v]:.3f})"
    return out

F_MAIN = "n_out ~ ln_pat + ln_base + C(corp_norm) + C(domain)"
F_NOBASE = "n_out ~ ln_pat + C(corp_norm) + C(domain)"
TOP4 = ["トヨタ自動車", "三菱電機", "日産自動車", "ホンダ"]
specs = [
    (d, F_NOBASE, "기준선 통제 없음"),
    (d, F_MAIN, "주 모형: 특허 + 기준선 논문"),
    (d[~d["corp_norm"].isin(TOP4)], F_MAIN, "상위 4개사 제외"),
    (d[d["n_win"] >= 30], F_MAIN, "특허 30건 이상 기업"),
    (d[d["domain"] != "POWER"], F_MAIN, "POWER 제외"),
    (d[d["domain"] != "VEHICLE"], F_MAIN, "VEHICLE 제외"),
    (d[d["n_pat"] >= 1], F_MAIN, "특허 1건 이상 칸만"),
]
res = []
for data, f, lab in specs:
    try:
        res.append(ppml(data, f, lab))
    except Exception as e:
        res.append({"model": lab, "error": str(e)[:100]})
print("\n[PPML 결과: ln_pat이 핵심 계수]")
print(pd.DataFrame(res).to_string(index=False))

In [ ]:
#

In [ ]:
import numpy as np, pandas as pd
import statsmodels.api as sm, statsmodels.formula.api as smf
pd.set_option("display.width", 200)

d = pd.read_csv("panel_firm_domain.csv")
d["as_pat"] = np.arcsinh(d["n_pat"]); d["as_base"] = np.arcsinh(d["n_base"])
d["ln_wo"] = np.log1p(d["n_wo"])

def ppml(data, formula, label, keys=("ln_pat", "as_pat", "ln_base", "as_base", "ln_wo")):
    x = data[data.groupby("corp_norm")["n_out"].transform("sum") > 0]
    x = x[x.groupby("domain")["n_out"].transform("sum") > 0]
    g = pd.factorize(x["corp_norm"])[0]
    r = smf.glm(formula, data=x, family=sm.families.Poisson()).fit(
        cov_type="cluster", cov_kwds={"groups": g}, maxiter=200)
    o = {"model": label, "firms": x["corp_norm"].nunique(), "cells": len(x)}
    for v in keys:
        if v in r.params.index:
            o[v] = f"{r.params[v]:.3f} (se {r.bse[v]:.3f}, p={r.pvalues[v]:.3f})"
    return o

FE = "+ C(corp_norm) + C(domain)"
specs = [
    (d, "n_out ~ as_pat + as_base " + FE, "asinh 변환"),
    (d[~d["domain"].isin(["LIGHT"])], "n_out ~ ln_pat + ln_base " + FE, "LIGHT 제외"),
    (d[~d["domain"].isin(["LIGHT", "BATTERY"])], "n_out ~ ln_pat + ln_base " + FE, "LIGHT·BATTERY 제외"),
    (d[d["n_pat"] >= 5], "n_out ~ ln_pat + ln_base " + FE, "특허 5건 이상 칸만"),
    (d, "n_out ~ ln_pat + ln_wo + ln_base " + FE, "WO 특허 통제 추가"),
]
print(pd.DataFrame([ppml(*s) for s in specs]).to_string(index=False))

In [ ]:
# **점검 결과는 일관됩니다. 기준선을 통제하면 특허의 추가 효과는 0.07~0.12 수준의 작은 값이고, 대부분 유의하지 않습니다.** 앞서 선택지 1을 기본으로 가시면 됩니다.

# ## 결과 해석

# | 점검 | ln_pat(또는 as_pat) | 판단 |
# |---|---|---|
# | 주 모형 (이전) | 0.078 (p=0.105) | 기준점 |
# | asinh 변환 | 0.074 (p=0.080) | 변환을 바꿔도 거의 같음 |
# | LIGHT 제외 | 0.095 (p=0.047) | 경계선 |
# | LIGHT·BATTERY 제외 | 0.072 (p=0.184) | 유의하지 않음 |
# | 특허 5건 이상 칸만 | **−0.029 (p=0.670)** | 0 |
# | WO 특허 통제 추가 | 0.124 (p=0.295) | 유의하지 않음 |

# 1. **0.07~0.12라는 크기가 변환과 표본 구성에 상관없이 유지됩니다.** 앞서 우려한 `log(1+x)` 민감성은 문제가 되지 않았습니다.
# 2. **p=0.047(LIGHT 제외)은 해석에 쓰지 마세요.** 이제 모형을 12개 가까이 돌렸고, 그중 하나가 0.05 근처에 걸린 것입니다. 같은 맥락에서 상위 4개사 제외의 p=0.017도 단독 근거로 쓰기 어렵습니다. 보고는 "추정치가 0.02~0.14 범위에서 대체로 양이지만 대부분 통계적으로 유의하지 않다"가 정직한 요약입니다.
# 3. **특허 5건 이상 칸만 쓰면 −0.029로 0이 됩니다.** 특허가 이미 충분히 있는 칸 안에서는, 특허가 더 많다고 논문이 더 늘지 않는다는 뜻입니다. 기준선 없는 모형의 0.476이 상당 부분 "특허가 0인 칸 대 양수인 칸"의 차이였다는 앞의 신호가 한 번 더 확인됩니다.
# 4. **WO 특허는 추가 정보가 없습니다**(−0.061, p=0.666). 해외 출원이 많은 분야일수록 논문이 더 나온다는 증거는 이 데이터에서 보이지 않습니다. 다만 WO 비중이 17.5%뿐이라 검정력이 낮습니다.

# ## 결론: 선택지 1로 마무리, 단 시간 순서 한계를 명시

# - 같은 방향의 결과가 여러 설정에서 반복되므로, 선택지 2(재수집, 호출 약 1,900회)를 해도 "효과가 작다"는 결론이 뒤집힐 가능성은 낮아 보입니다. 다만 이것은 추정이고, 시간 순서가 겹친 설계는 심사에서 지적받을 수 있습니다. 심사 대응이 필요하거나 지도교수님이 요구하시면 그때 추가하세요.
# - 기준선 통제가 "나쁜 통제"일 수 있다는 점은 그대로입니다. 기준선 없는 모형(0.476)과 있는 모형(0.078)을 **모두 표에 넣고** "진짜 효과는 두 값 사이에 있을 수 있다"고 논의에 쓰세요.

# ## 논문에 쓸 문장 초안

# **방법**
# > 기업 i의 기술 분야 d에 대해, 2006~2015년 공개 특허 수(ln(1+x))를 설명변수로, 2016~2025년 OpenAlex 논문 수를 결과변수로 하는 포아송 의사최우추정(PPML)을 추정하였다. 기업 및 분야 고정효과를 포함하였고, 표준오차는 기업 수준에서 군집화하였다. 2006~2015년 동일 기업·분야의 논문 수를 기준선 통제변수로 포함한 모형과 포함하지 않은 모형을 모두 보고한다.

# **결과**
# > 기준선 논문을 통제하지 않으면 특허와 이후 논문 사이에 강한 양의 연관이 나타났다(0.476, p<0.001). 그러나 기준선 논문을 통제하면 계수가 0.078로 줄어 통계적으로 유의하지 않았다(95% 신뢰구간 약 −0.02~0.17). 이는 특허 활동과 이후 논문 활동의 연관이 주로 기존 논문 활동의 지속성으로 설명됨을 시사한다. 변환 방식, 분야 구성, 표본 기준을 바꾼 강건성 분석에서도 추정치는 −0.03~0.14 범위에 머물렀다.

# **한계**
# > 기준선과 특허가 같은 기간을 공유하여 기준선 통제가 특허의 효과 일부를 흡수할 수 있다. 특허 자료는 등록된 문헌만 포함하며 2010~2015년이 91.8%를 차지한다. 기업 소속은 OpenAlex의 소속 매칭에 의존하며, 논문 분야는 `primary_topic` 하나만 사용하였다. 분야 대응표는 주제명 키워드 규칙에 수작업 수정을 더한 것이다.

# 위 신뢰구간(−0.02~0.17)은 se 0.048에서 제가 계산한 근사값이니, 논문에는 소프트웨어가 출력한 구간을 쓰세요.

# ## 남은 일

# - 시간 순서 문제를 정면으로 다루려면 선택지 2를 하세요. 하지 않아도 위 한계 문장으로 처리할 수 있습니다.
# - **분야별 이질성**을 보고 싶으시면(예: 어느 분야에서 특허-논문 연관이 큰가) 분야와 `ln_pat`의 상호작용을 넣는 모형을 만들어 드리겠습니다. 다만 분야별 기업 수가 17~52곳이라 검정력이 낮습니다.
# - 기업 단위 설계 A(횡단면)와 대학 공저 비중을 활용한 분석도 가능합니다.

# 어느 방향으로 이어가실지 알려 주세요.